- TODO: fix all todos
- TODO: review section headers
- TODO: how many steps is the ball invisible for
- TODO: measure RAM (filling up too much... previous run_envs?)
- TODO: ask gpt to look for typos and conceptual mistakes
- TODO: update "this notebook covers"
- TODO: update spec using entire notebook
- TODO: make all answer cells be separate
- TODO: timesteps vs steps
- TODO: does ball have velocity
- TODO: when does ball disappear
- TODO: manual vs supervised side by side for each env
- TODO: mention benefits of overfitting
- TODO: cluster frames by model activations
- TODO: bad loss due to 6 outpits

# **Beat Atari Pong - Part 1**

In this series, we'll tackle beating [Atari 2600 Pong](https://gymnasium.farama.org/environments/atari/pong/) using **Deep Reinforcement Learning**.

This notebook covers:

- Understanding the structure of Reinforcement Learning environments, focusing on [Gymnasium](https://gymnasium.farama.org/).
- Running **Atari 2600 Pong** via **Gymnasium** and interacting with it.
- Preprocessing observations to reduce complexity.
- Gathering a dataset using a manual policy and training a neural network to imitate that policy through supervised learning.

*Prerequisite: Basic knowledge of Neural Networks and PyTorch. For a deep-dive, explore these [notebooks](https://github.com/tsilva/aiml-notebooks/tree/main/karpathy-zero-to-hero).*

## **Introduction**

In this notebook series, we'll use **Gymnasium** to access **Pong** as a structured environment for implementing a reinforcement learning loop.

Gymnasium is a Python library that provides an API for interacting with environments. An **environment** is a world where an **agent** operates, following a cycle: the **agent** takes an **action**, receives an **observation** (information about the environment's state), and gets a **reward** (feedback based on the action). This process repeats throughout an **episode** until the environment terminates.

Key terms to know:

- **Environment**: The world in which the agent operates.
- **Agent**: The entity that interacts with the environment.
- **Observation**: Information the agent receives about the environment's current state.
- **State**: A complete representation of the environment's situation. In some environments, like **Pong**, the **observation** is equivalent to the **state**, providing all the information the agent needs for decision-making.
- **Action**: A decision made by the agent that affects the environment.
- **Reward**: Feedback from the environment based on the agent's action.
- **Episode**: A sequence of interactions from the initial state until the environment ends.
- **Step**: A single action-observation-reward cycle within an episode.
- **Policy**: A strategy the agent uses to choose actions based on observations, can be anything from an hardcoded script to a neural network.
- **Observation Space**: The structure and range of possible observations the agent can receive, defining what the environment can tell the agent at any given time.
- **Action Space**: The set of all possible actions the agent can take, defining the range of decisions the agent can make in the environment.

The goal is for the **agent** to explore, learn from **rewards**, and develop a **policy** that maximizes its total reward. To achieve this, we'll use **reinforcement learning** algorithms that help the agent understand which actions to take in each **state** and encode this strategy into a **policy**.

## **Setup**

Let's install the libraries we'll be using in the notebook:

In [ ]:
!pip -q install numpy matplotlib torch opencv-python-headless gymnasium ale-py

With **Gymnasium** installed, let's find the identifier for the **Pong** environment:

In [ ]:
import gymnasium as gym
import ale_py

# Make arcade learning environment available in gymnasium
gym.register_envs(ale_py)

# Retrieve list of available Pong environments
keys = gym.envs.registration.registry.keys()
[key for key in keys if "Pong" in key]

Several Pong environments are available, as listed in the [documentation](https://ale.farama.org/environments/pong/). Key differences include:

- **Observation Type**:
  - `obs_type=rgb`: Raw pixel data.
  - `obs_type=ram`: Byte data from game memory.

- **Frameskip**:
  - `frameskip=1`: Observes every game frame.
  - `frameskip=4`: Observes every 4th frame, reducing complexity.
  - `frameskip=(2, 5)`: Randomized between $2$ to $5$ frames (encourages learning of robust policy).

- **Repeat Action Probability**:
  - `repeat_action_probability=0.0`: Uses the action given at each step.
  - `repeat_action_probability=0.25`: 25% chance of repeating the previous action, adding randomness for robustness.

We'll use `PongNoFrameskip-v4` with `obs_type=rgb`, `frameskip=1`, and `repeat_action_probability=0.0` for raw pixel data and minimal complexity, to be as close as possible as to how a human interacts with the game.

---

Let's create utilities to monitor our experiments with the chosen environment.

We'll define a `render_video()` method to render and display a sequence of environment observations as a video. This method takes a list of frames (NumPy arrays in the shape `(height, width, channels)` and returns an embedded `HTML` object:

In [ ]:
import cv2
import base64
import imageio
import numpy as np
from io import BytesIO
from IPython.display import HTML

def render_video(
    frames,              # List of frames to render. Each frame can be an image or a tuple (image, label)
    scale=1,             # Scale factor for the video dimensions
    fps=30,              # Frames per second for the video
    format='mp4',        # Video format
    font_scale=0.5       # Font scale for text shown in video corner
):
    def adjust_frame_size(frame, block_size=16):
        """Resize frame dimensions to be divisible by block_size."""
        height, width = frame.shape[:2]
        new_height = (height + block_size - 1) // block_size * block_size
        new_width = (width + block_size - 1) // block_size * block_size
        return cv2.resize(frame, (new_width, new_height))

    # Define text properties once
    font = cv2.FONT_HERSHEY_SIMPLEX
    font_color = (0, 0, 0)  # Black
    line_type = 1
    margin = 10
    line_spacing = 20

    processed_frames = []
    for i, frame in enumerate(frames):
        # Determine if the frame includes a label
        if isinstance(frame, tuple) and len(frame) >= 2:
            image, label = frame
        else:
            image, label = frame, None

        # Adjust frame size
        image = adjust_frame_size(image)
        frame_with_text = image.copy()

        # If there's a label, render it
        if label:
            position = (margin, image.shape[0] - margin)
            font_color = (255, 255, 255) if len(frame_with_text.shape) == 2 else (0, 0, 0)
            cv2.putText(frame_with_text, label, position, font, font_scale, font_color, line_type)

        # Convert BGR to RGB for imageio
        _frame = frame_with_text
        if len(_frame.shape) == 2: _frame = cv2.cvtColor(frame_with_text, cv2.COLOR_GRAY2RGB)
        processed_frames.append(_frame)

    # Write video to buffer by appending frames individually
    buffer = BytesIO()
    with imageio.get_writer(buffer, format=format, fps=fps) as writer:
        for frame in processed_frames:
            writer.append_data(frame)

    # Encode video in base64
    video_data = base64.b64encode(buffer.getvalue()).decode()

    # Calculate scaled dimensions
    height, width = processed_frames[0].shape[:2]
    scaled_width, scaled_height = int(width * scale), int(height * scale)

    # Return HTML video tag
    return HTML(f'''
        <video width="{scaled_width}" height="{scaled_height}" controls autoplay loop muted>
            <source src="data:video/{format};base64,{video_data}" type="video/{format}">
            Your browser does not support the video tag.
        </video>
    ''')

# Case 1: Frames as images (no labels)
frames_images = [np.random.randint(0, 256, (128, 128), dtype=np.uint8) for _ in range(10)]
display(render_video(frames_images))

# Case 2: Frames as tuples (image, label)
frames_with_labels = [
    (np.random.randint(0, 256, (128, 128, 3), dtype=np.uint8), f"{i+1}")
    for i in range(10)
]
display(render_video(frames_with_labels))

To be able to compare different environment runs, we'll implement a `render_videos()` method. It accepts frame sequences paired with their respected video names (as tuples), rendering them side by side in a table for easy comparison:

In [ ]:
def render_videos(video_tuples, **kwargs):
    videos_frames = [x[0] for x in video_tuples] # Extract video frame lists from tuples
    videos = [x if isinstance(x, HTML) else render_video(x, **kwargs) for x in videos_frames] # Render videos if necessary
    labels = [x[1] for x in video_tuples] # Extract labels from tuples
    cell_style = "border: 2px solid grey; text-align: center; padding: 5px; font-weight: bold; text-transform: uppercase;" # The CSS for each cell
    video_row = "<tr>" + "".join([f"<td style='{cell_style}'>{video.data}</td>" for video in videos]) + "</tr>" # The row that shows the video
    label_row = "<tr>" + "".join([f"<td style='{cell_style}'>{label}</td>" for label in labels]) + "</tr>" # The row that shows the video's label
    return HTML(f"""
    <table style="width:100%; border-collapse: collapse; text-align: center;">
        {video_row}
        {label_row}
    </table>
    """)

# Test rendering three different videos side by side
render_videos([
    ([np.random.randint(0, 64, (64, 64, 3), dtype=np.uint8) for _ in range(10)], "Video 1"),
    ([np.random.randint(64, 128, (64, 64, 3), dtype=np.uint8) for _ in range(10)], "Video 2"),
    ([np.random.randint(128, 256, (64, 64, 3), dtype=np.uint8) for _ in range(10)], "Video 3"),
])

## **Inspect Environment**

Before building an agent, understanding the environment is crucial to identify its nuances, constraints, and challenges. We'll explore the **Observation Space**, **Action Space**, and **Environment Dynamics**, addressing key questions along the way. This will result in a detailed spec sheet that will guide us in solving this challenge.

---

### **Observations**

Let's start by exploring the **observation space** and answering the following questions:
  - **What observation data is provided, and how is it structured?**
  - **How frequently are observations updated?**
  - **Is there noise or irrelevant information in the observations?**

#### **Q: What observation data is provided, and how is it structured?**

To answer this question, let's initialize the environment and inspect its **observation space**:

In [ ]:
def make_env(env_id="PongNoFrameskip-v4"): return gym.make(env_id, render_mode="rgb_array")
env = make_env()
env.observation_space

The environment has a [Box](https://gymnasium.farama.org/api/spaces/fundamental/#box) observation space with:

- Shape: $(210, 160, 3)$
  - $210$: height
  - $160$: width
  - $3$: color channels (RGB)
- Values: 8-bit integers ranging from $0$ to $255$, representing pixel colors for each channel.

> `ANSWER:` **What observation data is provided, and how is it structured?** Observations are NumPy arrays with shape $(210, 160, 3)$—$210$ (height), $160$ (width), and $3$ (RGB channels). Values are 8-bit integers from $0$ to $255$, representing the intensity of each color channel per pixel.

---

#### **Q: How frequently are observations updated?**

Based on the [documentation](https://ale.farama.org/environments/pong/), we selected `PongNoFrameskip-v0` which has `frameskip=1`, meaning that each observation corresponds to a real game frame.



> `ANSWER:` **How frequently are observations updated?** Each timestep provides the next real image frame, with no frameskipping.

---

#### **Q: Is there noise or irrelevant information in the observations?**

To answer this one, we need to take a look at the actual game. Let's start by rendering the initial observation:

In [ ]:
env = make_env()
observation, _ = env.reset()
observation

The initial observation is unusual, with the left paddle and ball missing. We'll advance a few steps to detect when this changes.

Let's define a `run_env()` function to run the environment for a set number of steps and collect the data:

In [ ]:
def run_env(
    env,
    n_episodes=1, # Run for N episodes (optional, defaults to 1 episode)
    n_steps=None, # Run for N steps (optional, defaults to running until end of episode)
    action=None,  # Action to perform each timestep (optional, defaults to random action),
    seed=42       # Seed to reset the environment with (for reproducibility)
):
  # Reset the environment
  steps = 0
  observation, _ = env.reset(seed=seed)
  observations = [observation]

  # Run until end of episode or max steps reached
  for _ in range(n_episodes):
    terminated = False
    while not terminated and (n_steps is None or steps < n_steps):
      _action = action if action is not None else env.action_space.sample()
      observation, _, terminated, _, _ = env.step(_action)
      steps += 1
      observations.append(observation)

  # Return collected observations
  return observations

# Run for 1 step and return last observation (frame 2)
run_env(env, n_steps=1)[-1]

Frame $2$ still looks weird, let's run for one more step:

In [ ]:
run_env(env, n_steps=2)[-1] # Run for 2 steps and return last observation (frame 3)

It seems that after $2$ steps, the colors return to normal.

Let's run a few more steps:

In [ ]:
WARMUP_STEPS = 59
run_env(env, n_steps=WARMUP_STEPS)[-1] # Run for 59 steps and return last observation (frame 60)

After $59$ steps, at frame $60$, the opponent paddle and ball appear, marking the first proper game step.

Let's render a series of episodes to search for edge cases:

In [ ]:
render_videos([(run_env(env), label) for label in range(6)])

From observing random gameplay, we note:

- All environments were initialized with the same seed, yet they are all different (needs further investigation).
- The player paddle can be partially occluded at the top and bottom (unsure if it can be fully occluded).
- The opponent paddle can be fully occluded at the top and possibly the bottom (needs further investigation).
- The ball briefly disappears during wall bounces and after points (needs further investigation).
- Both paddles can keep moving after a point, even while the ball is invisible.
- The initial ball trajectory is consistent across points.
- After the opponent scores, the ball has the same initial trajectory over and over until the player is able to score back (unsure if this happens for opponent as well).

> `ANSWER:` **Q: Is there any noise or irrelevant information in the observations?**  
- The first two frames have color inaccuracies, but colors stabilize by step $3$.
- The opponent paddle and ball appear only from frame $60$ onward.
- White bars at the top and bottom can partially occlude both paddles.
- Opponent paddle can be fully occluded by the top white stripe (unclear if same is possible at bottom).
- The ball momentarily disappears during wall bounces and after each point.

---

### **Actions**

Now let's perform the same exploration for the **action space** and try to answer the following questions:
  - **Are actions discrete or continuous?**
  - **What actions can the agent take, and what are their effects?**
  - **Are there constraints on actions (timing, type)?**


---

#### **Q: Are actions discrete or continuous?**

We can inspect the environment object to know details about the action space:

In [ ]:
env.action_space

To generate a list with all possible actions you can do the following:

In [ ]:
ACTIONS = list(range(env.action_space.start, env.action_space.n))
ACTIONS

> `ANSWER:` **Q: Are actions discrete or continuous?**
The action space is discrete, and these are the available actions: $[0, 1, 2, 3, 4, 5, 6]$.


---

#### **Q: What actions can the agent take, and what are their effects?**

To answer this question, we should generate an episode for each action and perform only that action throughout the episode:

In [ ]:
render_videos(
    [(run_env(env, n_steps=100, action=action), action) for action in ACTIONS]
)

> `ANSWER:` **Q: What actions can the agent take, and what are their effects?** Here's what each action does:
- `0`: Does nothing  
- `1`: Does nothing (same as `0`)    
- `2`: Moves the right paddle up  
- `3`: Moves the right paddle down  
- `4`: Moves the right paddle up (same as `2`)  
- `5`: Moves the right paddle down (same as `3`)  

---

#### **Q: Are there constraints on actions (timing, type)?**

Given the quirky initial frames, where colors are off and the ball and opponent paddle are missing, it's worth exploring whether the paddle can move during those steps. Before we start our experiments, let's create an enumeration with the actions we'll be using:



In [ ]:
from enum import Enum

class Actions(Enum):
    NOOP = 1
    UP = 2
    DOWN = 3

ACTIONS = [action.value for action in Actions]
ACTIONS

Let's try moving down right from the beginning:

In [ ]:
render_video(run_env(env, n_steps=4, action=Actions.DOWN.value), fps=4)

It seems the paddle can be moved right from the start of the episode, so even the initial frames with incorrect colors and missing opponent paddle and ball still have value.

---

Since we're using `PongNoFrameskip-v4` with `repeat_action_probability=0.0`, we expect that the action we request at each step is the action that is actually performed, and that the resulting observation accurately reflects that action.

Let's confirm if that expectation is true:

In [ ]:
def _run_env_actions(actions, seed=42):
  # Initialize the environment
  # and collect first frame
  env = make_env()
  frames = [env.reset(seed=seed)[0]]

  # Execute actions and collects frames
  frames = []
  for index, action in enumerate(actions):
    observation = env.step(action)[0]
    frames.append((observation, str(index)))

  # Return frames
  return frames

# Create action queue
actions = [Actions.NOOP.value] * WARMUP_STEPS # Perform NOOP until paddles and ball appear
actions += [Actions.DOWN.value] * 4           # Move DOWN 4 times
actions += [Actions.UP.value] * 4             # Move UP 4 times
actions += [Actions.NOOP.value] * 20          # Perform 20 NOOP steps to pad the video with more frames

# Render frames as video at 1 fps, with frame numbers
# visible, to debug if each frame matches the action
render_video(_run_env_actions(actions), fps=1)

By analyzing the video, we observe:

- The ball moves one pixel per frame.
- The opponent's paddle updates every second frame, moving one pixel each time.
- The player paddle updates on alternating frames from the opponent, so they never move simultaneously. However, the player paddle's movement is inconsistent, suggesting that observations may not always accurately reflect the actions taken.

This inconsistency raises the question of whether all actions are being executed. It could be due to input processing delays, where the environment uses outdated observations, or it might be discarding some actions. While neither is a major issue for training, the latter could be more problematic.

To verify that all actions are executed, we'll conduct this experiment:

1. Move the paddle to the bottom using the minimum number of actions possible.
2. Save this frame as `frame1`.
3. Move the paddle to the top using the minimum number of actions possible.
4. Move it back to the bottom.
5. Repeat steps 3 and 4 several times.
6. Perform extra `NOOP` actions to ensure all movements are rendered.
7. Save the final frame as `frame2`.
8. Subtract `frame1` from `frame2` and display the result.

If no paddle appears in the subtraction of both frames, it confirms that the paddle returned to its original position, indicating that all actions were executed correctly.

To be able to perform the experiment, first we need to determine how many steps are necessary to reach the edges from the center:

In [ ]:
actions = [Actions.DOWN.value] * 16 + [Actions.NOOP.value] * 4
render_video(_run_env_actions(actions), fps=1)

From the center, we need to perform the `DOWN` action $16$ times, followed by at least $4$ `NOOP` actions to see the result. These values were determined through trial and error. Feel free to experiment by reducing the number of `DOWN` or `NOOP` actions, and re-running the cell.

Now we need to determine how many steps are required to move from one edge to the other. To test this, we'll first move the paddle to the bottom of the screen, then perform `UP` actions:

In [ ]:
reset_actions = [Actions.DOWN.value] * 16 + [Actions.NOOP.value] * 4
actions = reset_actions + [Actions.UP.value] * 28 + [Actions.NOOP.value] * 4
render_video(_run_env_actions(actions), fps=1)

From the edge, we need to perform the `UP` action $28$ times, followed by at least $4$ `NOOP` actions to see the result. These values were determined through trial and error. Feel free to experiment with reducing the number of `UP` or `NOOP` actions.

If our interpretation of the asynchrony between actions and observations is correct, removing the `NOOP` actions after the `DOWN` actions might prevent us from seeing the paddle reach the bottom (instead showing a mid-frame between moving down and up). However, we should still see it reach the top edge since that is followed by `NOOP` actions. Let's confirm:

In [ ]:
reset_actions = [Actions.DOWN.value] * 16
actions = reset_actions + [Actions.UP.value] * 28 + [Actions.NOOP.value] * 4
render_video(_run_env_actions(actions), fps=1)

The behavior was exactly as expected, so we're now ready to run the experiment to determine if all actions are being processed:

In [ ]:
MAX_STEPS_FROM_EDGE = 28  # Max steps to move the paddle from the edge to center
MAX_STEPS_FROM_CENTER = 16  # Max steps to move the paddle from center to edge
NOOP_FRAMESKIP = [Actions.NOOP.value] * 4 # List of actions to perform after a real action to ensure that the next frame we see displays the previous actions fully executed, as far as we can tell, we need 4 frames to flush out the results of each action
NUM_REPS = 10  # Number of repetitions for the back-and-forth movement

# Run environment with the action queue we require for this experiment
reset_actions = ([Actions.DOWN.value] * MAX_STEPS_FROM_CENTER) + NOOP_FRAMESKIP
extra_actions = ((([Actions.UP.value] * MAX_STEPS_FROM_EDGE) + ([Actions.DOWN.value] * MAX_STEPS_FROM_EDGE)) * NUM_REPS) + NOOP_FRAMESKIP
actions = reset_actions + extra_actions
frames = _run_env_actions(actions)

# Render the video alongside key frames
render_videos([
    (frames, "Video"), # Render the whole video
    ([frames[len(reset_actions)][0]], "First Frame"), # Render the first frame, right after the paddle has finished moving from the center to the bottom
    ([frames[-1][0]], "Last Frame"), # Render the last frame, right after the paddle has finished moving between both edges multiple times
    ([frames[-1][0] - frames[len(reset_actions)][0]], "Frame Diff"), # Render the difference between the last and the first frame
], fps=1)

From the experiment, we observe:

- The paddle starts and ends at the bottom position.
- The difference between the first and last frames does not show the paddle, indicating that after moving between both edges multiple times, it returned to the exact same initial position.
- During playback, the paddle's movements to the edge positions are not visible in the intermediate frames.

Therefore, we can conclude that:

- Actions are executed correctly by the environment.
- A slight desynchronization exists between action processing and observation updates.
- This means a `DOWN` action might not immediately appear in observations; a quick `UP` action could show the paddle moving up without visibly reaching the bottom.

---

We should also check if the action space sampling functions correctly, produces a uniform distribution, and allows setting a seed for reproducible results:

In [ ]:
for _ in range(2):
  env = make_env()
  print([env.action_space.sample() for _ in range(5)])

If you create an environment, sample from its action space, create another, and sample again, the samples differ. This suggests that the action space sampler does not use a random number generator initialized with a specific seed.

The `reset()` method accepts a seed—let's test it that way:

In [ ]:
for _ in range(2):
  env = make_env()
  env.reset(seed=42)
  print([env.action_space.sample() for _ in range(5)])

No luck with `reset()`. It seems there's an `env.action_space.seed()` method—let's try that instead:

In [ ]:
for _ in range(2):
  env = make_env()
  env.action_space.seed(42)
  print([env.action_space.sample() for _ in range(5)])

It worked! To ensure reproducible action space sampling, set a seed using `env.action_space.seed()`. Earlier, episodes differed because the action space wasn't seeded. Let's rerun the experiment and confirm the episodes are now consistent:

In [ ]:
videos = []
for label in range(6):
  env.action_space.seed(42)
  videos.append((run_env(env, seed=label), label))
render_videos(videos)

As expected, the episodes are now identical, confirming that episode randomness is entirely driven by the policy's randomness.

---

One final check: let's confirm that sampling generates a uniform distribution. This is crucial, as any bias in the sampling method could undermine our training process without us realizing it.

In [ ]:
import matplotlib.pyplot as plt

values = [env.action_space.sample() for _ in range(10_000)] # Sample N actions
bins = [i - 0.5 for i in range(env.action_space.start, env.action_space.start + env.action_space.n + 1)] # Create a bin for each possible action
plt.hist(values, bins=bins, edgecolor='black') # Plot sample frequency of each possible action

The sampled actions form a uniform distribution. You can run the cell multiple times to see different histograms, but they will always approximate a uniform distribution. The more actions you sample, the closer the results will be to a true uniform distribution.

> `ANSWER:` **Q: Are there constraints on actions (timing, type)?**
- All actions are processed, but their effects may not appear immediately in observations. It can take up to two `NOOP` steps for the observation to fully reflect the outcome of an action. Without `NOOP` steps, intermediate frames might blend actions (e.g., moving down then up could show a frame between these movements).
- `env.action_space.sample()` generates uniformly distributed actions and is reliable. For reproducibility, set a seed using `env.action_space.seed()`.

---

### **Environment Dynamics**

Let's explore the **environment dynamics** by answering the following questions:

- **What are the goals and how are rewards assigned?**
- **How long is each episode, and what ends it?**
- **Is there any difficulty progression?**
- **Is randomness present in initial conditions, observations, or outcomes?**
- **Are there patterns or quirks affecting performance or training?**


---

#### **Q: What are the goals and how are rewards assigned?**

According to the [documentation](https://ale.farama.org/environments/pong/):

> You control the right paddle, competing against the left paddle (computer-controlled). Both players aim to deflect the ball into the opponent's goal while protecting their own.

We need to understand how rewards are distributed at each step, the possible total reward range for an episode, and the agent's goal, which is to maximize its score while minimizing the opponent's. The game ends when a player reaches $21$ points.

Let's render a full game, displaying the reward received in each frame, the total reward, and the types of rewards received and their frequency:

In [ ]:
import sys

# Initialize environment (specify seed that results
# in player scoring at least one point)
env = make_env()
env.reset(seed=111)
env.action_space.seed(111)

# Run until episode termination
frames, rewards_map, terminated, episode_step, episode_reward = [], {}, False, 0, 0
while not terminated:
  observation, reward, terminated, _, _ = env.step(env.action_space.sample()) # Perform random action
  rewards_map[reward] = rewards_map.get(reward, 0) + 1 # Count occurrency of reward
  episode_step += 1
  episode_reward += reward # Add to episode reward
  frames.append((observation, f"{episode_step}: {reward};{episode_reward}")) # Collect frame for later rendering video (label it with step, step reward, and episode reward)

# Print the reward types and occurrences
print(rewards_map)

# Render the video slowly in order to be
# able to understand when rewards occur
render_video(frames, fps=1)

By analyzing the video, we observe:

- Reward is $0$ when no events occur.
- At step $256$, the opponent scores its first point, resulting in a reward of $-1$.
- At step $2373$, the player scores, resulting in a reward of $1$.
- The game ends with a total reward of $-20$.

From this, we can conclude:

- Scoring a point gives a reward of $1$.
- Conceding a point gives a reward of $-1$.
- The total reward for an episode ranges from $[-21, 21]$.
  - A score of $21$ means the player wins without conceding.
  - A score of $-21$ means the player loses without scoring.
  - Other values reflect a combination of points scored and conceded.
- Rewards are $0$ for all other steps.
- There seem to be no secondary objectives.

> `ANSWER:` **What are the goals and how are rewards assigned?** The environment's goal is to maximize the score by earning points and minimizing points conceded. Scoring gives a reward of $1$, conceding gives $-1$, and all other steps yield $0$. Total rewards range from $-21$ (loss without scoring) to $21$ (win without conceding), with other scores reflecting a mix of outcomes. There are no secondary objectives.

---
#### **Q: How long is each episode, and what ends it?**

This question was mostly addressed earlier. The only remaining uncertainty is whether an episode is truncated if it lasts too long. To test this, we would need a policy that deflects the ball indefinitely without scoring, allowing us to see if the episode terminates before a player reaches $21$ points.

However, creating such a policy is currently impractical and might not even be possible. Therefore, it's reasonable to assume that an episode will continue as long as necessary until one player scores $21$ points.

> `ANSWER:` **How long is each episode, and what ends it?**
Each episode ends when a player reaches $21$ points, with the duration depending on how quickly points are scored. It's unclear if episodes can be truncated for being too long.

---
#### **Q: Is there any difficulty progression?**

Our previous explorations suggest that the difficulty remains constant throughout the episode. The documentation mentions additional [difficulty modes](https://ale.farama.org/environments/pong/#difficulty-and-modes) that can be set during environment creation, but it's unclear if they have an effect. The best way to test this is to try harder difficulties after training a policy that succeeds at the default level.

> `ANSWER:` **Is there any difficulty progression?**
There is no difficulty progression, difficulty is the same from beginning to end of episode.

---
#### **Q: Is randomness present in initial conditions, observations, or outcomes?**

We already have a strong suspicion that the randomness of the environment is fully determined by the randomness of the policy, however, let's perform some experiments to confirm this.

The first exploration worth doing is if the environment starts differently when initialized with different seeds. Namely if the ball is shot in a different direction and/or if the paddles start in different locations:

In [ ]:
render_videos([(run_env(env, action=Actions.NOOP.value, seed=seed), seed) for seed in range(4)], fps=10)

By analyzing the video, we observe:

- Paddles start in the same position, regardless of the seed.
- The game starts at the same step, regardless of the seed.
- The ball is initially shot in the same direction, regardless of the seed.
- After the first point is scored, the ball is shot in a different, but consistent direction, regardless of the seed.
- From the second point onward, the ball is always shot in the same direction, regardless of the seed.

Thus, we can conclude that the environment is entirely deterministic, and the seed has no influence on it. Any randomness comes solely from the stochastic nature of the policy.



Let's triple check the theory by creating a map of the ball trajectories throughout the episode. First

In [ ]:
frames = run_env(env, action=Actions.NOOP.value) # Run env and collect frame list
clean_frames = frames[WARMUP_STEPS:] # Discard quirky frames
clean_frames_np = np.array(clean_frames) # Create numpy array to have a single structure that has a temporal dimension
clean_frames_np.shape

Now that the frames are stored in a NumPy array, we can collapse the temporal dimension (dimension $0$) into a single frame by taking the maximum value across all instances. Since the ball's color is the maximum value, $255$, this process should highlight the ball's trajectory:

In [ ]:
clean_frames_np.max(0)

It worked let's package this into an util and run it for multiple environments:

In [ ]:
def _run_env_max(env, **kwargs):
  frames = run_env(env, **kwargs) # Run env and collect frame list
  clean_frames = frames[WARMUP_STEPS:] # Discard quirky frames
  clean_frames_np = np.array(clean_frames) # Create numpy array to have a single structure that has a temporal dimension
  return [clean_frames_np.max(0)] # Return video with just frame (max of all frames)

# Run 6 environments with different seeds for 3 episodes each, and render the max frame from each environment side by side
render_videos(
    [(_run_env_max(env, n_episodes=3, action=Actions.NOOP.value, seed=seed), seed) for seed in range(4)]
)

Despite the different seeds and running multiple episodes, all environments turned out the same, reinforcing the deterministic nature of this environment.

An additional exploration worth conducting is to determine if the mismatch between actions and resulting observations, due to input lag, is consistent across episodes and environments with different seeds. To do this, we can use the same action sequence from a previous experiment to bounce the paddle between both edges, apply this sequence in environments with different seeds, and compare the results side by side.

In [ ]:
# Create action sequence that moves paddle up and down between both edges
reset_actions = ([Actions.DOWN.value] * MAX_STEPS_FROM_CENTER) + NOOP_FRAMESKIP
extra_actions = ((([Actions.UP.value] * MAX_STEPS_FROM_EDGE) + ([Actions.DOWN.value] * MAX_STEPS_FROM_EDGE)) * NUM_REPS) + NOOP_FRAMESKIP
actions = reset_actions + extra_actions

# Run the actions in 4 environments with different seeds and render them
video_tuples = [(_run_env_actions(actions, seed=seed), seed) for seed in range(4)]
render_videos(video_tuples)

They all look identical, but to be certain, we can hash all the frames from each video. If the videos are pixel-for-pixel identical, their hashes will match:

In [ ]:
[hash(np.array([frame for frame, label in video_tuple[0]]).tobytes()) for video_tuple in video_tuples]

All the videos have identical hashes, indicating that the asynchrony between input and resulting observations is consistent across environments, even with different seeds.

> `ANSWER:` **Is randomness present in initial conditions, observations, or outcomes?**
No, randomness is not present in the environment's initial conditions, observations, or outcomes. The starting positions and ball movements are consistent regardless of the seed. Randomness occurs only if the actions themselves have stochastic properties.

---
#### **Q: Are there patterns or quirks affecting performance or training?**

From our explorations, we identified these issues:

- Initial frames have unusual colors.
- The ball's trajectory is consistent after points, leading to repeated scenarios.
- There is a delay between actions and their effects in observations.
- The ball disappears briefly after a point.
- Paddles can be obscured by white bars.
- Rewards are sparse, mostly zero.
- The environment is deterministic.

We conclude that:

- Initial frames need color correction.
- The delay between actions and observations may add noise, but they are predictable, so they shouldn't interfere with training.
- Consistent ball direction after scoring can cause repeated failures, making most of the collected training data be redundant when the policy fails to recover.
- Environment determinism will make overfitting likely.
- Sparse rewards will make learning slow.



> `ANSWER:` **Are there patterns or quirks affecting performance or training?**
Yes, factors such as initial frame colors, consistent ball trajectories, and delays between actions and observations can impact training. Determinism risks overfitting, while sparse rewards can slow down learning.

### **Environment Summary**

We have finished our inspection, here is a summary of what we found:

- **Observation Space:** $(210, 160, 3)$ RGB pixels, values range $[0,255]$.
- **Action Space:** Discrete, 6 actions $[0, 1, 2, 3, 4, 5]$; key actions are `1` (NOOP), `2` (Up), `3` (Down).
- **Rewards:** `+1` for player scoring, `-1` for opponent scoring; all other steps yield `0`.
- **Episode End:** Game ends when either player scores `21` points.
- **Initial Frames:** Colors stabilize after frame `3`; paddles and ball appear from frame `60`.
- **Frameskip:** No frameskipping, each frame is a direct game observation.
- **Action Effects:** Observations may take up to $2$ extra frames to fully reflect actions, adding noise but encouraging robustness.
- **Ball Behavior:** After scoring, ball follows a consistent trajectory, leading to repetitive gameplay.
- **Preprocessing Suggestions:** Crop out scoreboard, use binary image representation, focus on paddles and ball.
- **Randomness:** Use `env.action_space.seed()` for consistent action sampling and initial conditions.
- **Complexity Reduction:** Simplify observation to $2$ colors (paddle/ball vs. background) for faster training.

## **Reduce Environment Dimensionality**

Solving this environment with reinforcement learning, without prior knowledge, is challenging due to high dimensionality and sparse rewards.

Each observation has a resolution of $160 \times 210 \times 3$ with pixel values in the range $[0, 255]$, leading to up to $25,704,000$ possible unique observations. With $6$ actions, this results in $154,224,000$ potential observation-action transitions. However, the true observation space is smaller, as not all colors appear in every pixel, and some actions are redundant.

Instead of training on the full observation and action space immediately, it's more practical to reduce dimensionality, simplifying the problem. This allows for easier solutions initially, with complexity gradually increased. It also makes it easier to track RL algorithm progress, especially in environments with high-dimensional image data.

Now, let's simplify the observations by removing the score at the top. Since the score isn't needed for a winning strategy, keeping it only inflates the observation space. Any ball and paddle combinations can occur with any score, and with a maximum score of $20/20$, this could increase the observation space by a factor of $400$ ($20 \times 20 = 400$).

Let's truncate the observation to remove the scoreboard and the white bar below it:

In [ ]:
env = make_env()
env.reset()
for _ in range(WARMUP_STEPS): observation = env.step(Actions.NOOP.value)[0]

OBS_TOP_Y = 35
observation[OBS_TOP_Y:, ::, ::]

The bottom bar is also useless, let's truncate it as well:

In [ ]:
OBS_BOTTOM_Y = -16
observation[OBS_TOP_Y:OBS_BOTTOM_Y, ::, ::]

The space behind the paddles is also irrelevant, as once the ball is behind the paddle, the outcome is already determined. Let's strip that out as well:

In [ ]:
OBS_LEFT_X = 19
OBS_RIGHT_X = -19
observation[OBS_TOP_Y:OBS_BOTTOM_Y, OBS_LEFT_X:OBS_RIGHT_X, ::]

We have $3$ color channels, but using just one will still allow us to observe everything important about the game. Let's use only the blue color channel:

In [ ]:
OBS_CHANNEL = 2
observation[OBS_TOP_Y:OBS_BOTTOM_Y, OBS_LEFT_X:OBS_RIGHT_, OBS_CHANNEL]

Most colors are unnecessary. We can simplify by using just two: one for the background and one for the paddles and ball. To replace the colors, we first need to inventory the current color codes to be able to target and replace them:

In [ ]:
import numpy as np
observation, _ = env.reset()
for _ in range(WARMUP_STEPS): observation, _, _, _, _ = env.step(Actions.NOOP.value)
values = np.unique(observation[OBS_TOP_Y:OBS_BOTTOM_Y, OBS_LEFT_X:OBS_RIGHT_X, OBS_CHANNEL])
values, plt.imshow([values], cmap="gray")

Color `17` is the background, we can turn that black, and all others white. However, we need to make sure we check this for the initial quirky frames as well, since these have different colors:

In [ ]:
observation, _ = env.reset()
values = np.unique(observation[OBS_TOP_Y:OBS_BOTTOM_Y, OBS_LEFT_X:OBS_RIGHT_X, OBS_CHANNEL])
values, plt.imshow([values], cmap="gray")

Ok, it seems we should convert both colors `17` and `43` to black and all others to white:

In [ ]:
def preprocess_frame(frame):
  frame = frame[OBS_TOP_Y:OBS_BOTTOM_Y, OBS_LEFT_X:OBS_RIGHT_X, OBS_CHANNEL]
  frame[(frame != 17) & (frame != 43)] = 255
  frame[(frame == 17) | (frame == 43)] = 0
  return frame

initial_observation = env.reset()[0]
frames = [preprocess_frame(initial_observation)]
while True:
    observation, reward, terminated, truncated, info = env.step(Actions.NOOP.value)
    frames.append(preprocess_frame(observation))
    if terminated: break
values = np.unique(frames)
print(values)
render_video(frames, fps=1)

With only $2$ colors, a height of $159$, a width of $122$, and $3$ possible actions, unique observations reduce to $2 \times 159 \times 122 \times 3 = 116,388$. Starting from $154,224,000$, we've reduced the exploration space by $1000$x. In practice, the observation space is even smaller, as many configurations (e.g., all white pixels or fully occupied columns) are impossible, further simplifying the problem.

Let's store the dimensions of a preprocessed observation for later usage:

In [ ]:
PREPROCESSED_OBS_HEIGHT, PREPROCESSED_OBS_WIDTH = frames[0].shape
PREPROCESSED_OBS_HEIGHT, PREPROCESSED_OBS_WIDTH

Now we should confirm that frame preprocessing is working fine by running the game for a while and checking the resulting video. We'll take the opportunity to revamp the `run_env()` method to make it more flexible:

In [ ]:
# TODO: add stats for multiple episodes, namely means

import time

SEED = 42
OBS_MAP = {}

def load_obs(obs_hash):
  return OBS_MAP[obs_hash]

def store_obs(obs):
    obs_hash = hash(obs.tobytes())
    OBS_MAP[obs_hash] = obs
    return obs_hash

def run_env(
    env,
    n_episodes=1,
    n_steps=None,
    preprocessor=None,
    policy=None,
    reset_when_finished=True,
    seed=SEED
):
    # Clear the observations map
    OBS_MAP.clear()

    # The policy to use when choosing an action to perform on the
    # environment each timestep (defaults to sampling a random action)
    if policy is None: policy = lambda _: env.action_space.sample()
    elif not callable(policy): action = policy; policy = lambda _: action

    # The trajectories for all episodes
    # (a trajectory is the sequence of observations, steps,
    # rewards and other info obtained at each time step during an episode)
    trajectories = []

    for episode in range(n_episodes):
        # Reset the environment
        if episode == 0 or reset_when_finished:
            obs, info = env.reset(seed=seed)
            obs_hash = store_obs(obs)

            # Preprocess the observation (usually to reduce the dimensionality
            # of the observation, like downsampling or converting to grayscale)
            obs_processed = preprocessor(obs) if preprocessor else obs
            obs_processed_hash = store_obs(obs_processed)

        # Start a time counter for this episode
        start_time = time.time()

        # The number of steps performed in this episode so far
        episode_step = 0

        # The total reward collected this episode
        episode_reward = 0

        # Initialize the trajectory for this episode
        episode_trajectory = []

        # While the maximum number of steps is not reac
        while True:
            # Use the policy to select which action should be taken
            # based on the observation that was observed this time step
            action = policy(obs_processed)

            # Perform the action on the environment and collect feedback
            next_obs, reward, terminated, _, _ = env.step(action)
            episode_step += 1

            # Store the next observation
            next_obs_hash = store_obs(next_obs)

            # Preprocess the next observation and hash it
            next_obs_processed = preprocessor(next_obs) if preprocessor else next_obs
            next_obs_processed_hash = store_obs(next_obs_processed)

            # Add the step data to the episode trajectory
            episode_trajectory.append({
                "obs_hash" : obs_hash,
                "obs_processed_hash" : obs_processed_hash,
                "action" : action,
                "reward" : reward,
                "next_obs_hash" : next_obs_hash,
                "next_obs_processed_hash" : next_obs_processed_hash,
                "terminated" : terminated
            })

            # Add the step reward to the total episode reward
            episode_reward += reward

            obs = next_obs
            obs_hash = next_obs_hash
            obs_processed = next_obs_processed
            obs_processed_hash = next_obs_processed_hash

            # In case the environment said that it has terminated then end the episode
            if terminated:
                break

            # In case a max number of steps was specified and reached then end the episode
            if n_steps and episode_step >= n_steps:
                break

        # Add the episode trajectory to the trajectory list
        trajectories.append(episode_trajectory)

        # Log episode stats
        episode_duration = time.time() - start_time
        stats = {
            "episode": episode,
            "steps": episode_step,
            "reward": episode_reward,
            "duration": f"{episode_duration:.2f}s"
        }
        print(stats)

    # Return the trajectories collected for all episodes
    return trajectories

trajectories = run_env(
    env,
    preprocessor=lambda x:preprocess_frame(x)
)
collect_frames_fn = lambda trajectories, key, label=True: [(load_obs(step[key]), str(index)) if label else load_obs(step[key]) for episode in trajectories for index, step in enumerate(episode)]
render_videos([
    (collect_frames_fn(trajectories, key, label=True), key) for key in ["obs_hash", "obs_processed_hash", "next_obs_hash", "next_obs_processed_hash"]
], fps=2)

# TODO: mention that opponent paddle can get out of range

Training a policy directly from pixels is challenging because the model must learn both actions and feature extraction using only sparse rewards. In this environment, the agent will struggle to learn until it scores a point by chance, as all actions taken until then will be seen as failures.

To avoid a prolonged, unclear training phase, we'll simplify the problem by extracting features, bootstrapping a model with a basic policy, and using that as a foundation. Here's the plan:

1. **Extract Features**: Write code to extract `left_paddle_y`, `right_paddle_y`, `ball_x`, and `ball_y`.
2. **Create Basic Policy**: Implement a policy that uses the extracted features to track the ball's position.
3. **Create Features-Action Dataset**: Run episodes using the basic policy, collect features-action pairs, and create a dataset.
4. **Train Basic Policy Model (features input)**: Train a model to mimic the basic policy using the collected dataset.
5. **Test Reinforcement Learning**: Apply a reinforcement learning algorithm to this model, using extracted features as input for faster training and easier debugging.
6. **Create Image-Action Dataset**: Run episodes with the basic policy and collect image-action pairs to build a dataset.
7. **Train Basic Policy Model (image input)**: Train a model using the new image-action dataset.
8. **Fine-Tune with RL**: Fine-tune this model with reinforcement learning, using image-based inputs.

This approach provides a smoother learning curve, building up from simpler inputs to more complex image-based training.

Let's start by extracting the features. For the left paddle, we'll find its position by identifying the first white pixel in the first column:

In [ ]:
trajectories = run_env(
    env,
    n_steps=WARMUP_STEPS+1,
    preprocessor=lambda x:preprocess_frame(x),
    policy=Actions.NOOP.value
) # Run steps until both paddles appear
first_episode_trajectory = trajectories[0] # Retrieve episode trajectory
first_valid_step = first_episode_trajectory[WARMUP_STEPS] # Retrieve first step where both paddles appear
frame = load_obs(first_valid_step["obs_processed_hash"]) # Load the processed observation for that step (preprocessed black and white frame)
frame[:, 0] # Return first column

Notice those sequential `255` values? That's the left paddle, we only need to grab its index:

In [ ]:
left_paddle_y = np.argmax(frame[:, 0] == 255) # Retrieve index of first value in leftmost column that is white (topdown)
assert frame[left_paddle_y, 0] == 255 # Assert that there is indeed a white pixel in the retrieved position
left_paddle_y # Display retrieved position

The left paddle is at index `80`. Now let's do the same for the right paddle:

In [ ]:
right_paddle_y = np.argmax(frame[:, -1] == 255) # Retrieve index of first value in rightmost column that is white (topdown)
assert frame[right_paddle_y, -1] == 255 # Assert that there is indeed a white pixel in the retrieved position
right_paddle_y # Display retrieved position

The right paddle is at index `61`. Now we need to retrieve the ball's position. To do so, we're going to strip out the columns where the paddles are. With that done, any white pixel we find in the remaining area has to belong to the ball:

In [ ]:
ball_search_space = frame[:, 1:-2]
ball_y, ball_x = np.where(ball_search_space == 255)
assert len(ball_y) > 0 and len(ball_x) > 0, "No ball found"
ball_x, ball_y = ball_x[0], ball_y[0]
ball_x, ball_y

And the ball is at `x = 58` and `y = 80`. Now that we know how to extract the features, let's create an extraction method, make it part of the preprocessing pipeline, and record an episode where we overlay the extracted features over the episode so that it's easy to spot any features we have extracted incorrectly:

In [ ]:
def extract_features(frame):
  # Retrieve left paddle position
  left_paddle_y = np.argmax(frame[:, 0] == 255)
  if frame[left_paddle_y, 0] == 0: left_paddle_y = -1

  # Retrieve right paddle position
  right_paddle_y = np.argmax(frame[:, -1] == 255)
  if frame[right_paddle_y, -1] == 0: right_paddle_y = -1

  # Retrieve ball position
  ball_search_space = frame[:, 1:-2]
  ball_y, ball_x = np.where(ball_search_space == 255)
  ball_x, ball_y = (ball_x[0], ball_y[0]) if len(ball_y) > 0 and len(ball_x) > 0 else (-1, -1)

  # Package features into array and return it
  features = np.array([left_paddle_y, right_paddle_y, ball_x, ball_y])
  return features

# Run an episode
trajectories = run_env(
    env,
    preprocessor=lambda x: extract_features(preprocess_frame(x))
)

# Collect images
frames = collect_frames_fn(trajectories, "obs_hash", label=False)

# Collect features
features2str = lambda x: np.array2string(x, precision=2, separator=', ', floatmode='fixed')
features = [f"{x[1]}: {features2str(x[0])}" for x in collect_frames_fn(trajectories, "obs_processed_hash")]

# Bundle images with corresponding features
frames_tuples = list(zip(frames, features))

# Render video
render_video(frames_tuples, fps=2, font_scale=0.25)

## **Manual Policy**

Let's manually implement a policy that will be pretty decent at the game. It's pretty straightforward. We just need to make the paddle track the ball and that should be good enough to be a decent policy. If the ball is within the paddle's length, the paddle doesnt move, if its above, it moves up, if its below, it moves down. The only edge case is that if the ball is right on the edge of the paddle, and close to the goal line, it can manage to get past the paddle. To mitigate this risk, instead of considering the paddle's length as the range when the paddle doesnt move while the ball is within, we should pretend that the paddle is smaller, that way its encouraged to move in the direction of the ball before it reaches the paddle.

Let's try it out with a regular paddle:

In [ ]:
RIGHT_PADDLE_LENGTH=16 # TODO: move this out of here

def manual_policy(features, threshold=0.25, right_paddle_length=RIGHT_PADDLE_LENGTH):
    left_paddle_y, right_paddle_y, ball_x, ball_y = features
    if -1 in (right_paddle_y, ball_x, ball_y): return Actions.NOOP.value

    # Calculate top and bottom y positions for the paddle while
    # trimming it on the sides to encourage the paddle to move earlier
    right_paddle_top_y = right_paddle_y + right_paddle_length * threshold
    right_paddle_bottom_y = right_paddle_y - right_paddle_length * threshold

    # If ball is within trimmed paddle boundaries, don't move,
    # otherwise move up/down to get ball within boundaries
    action = Actions.NOOP.value
    if ball_y < right_paddle_bottom_y: action = Actions.UP.value
    elif ball_y > right_paddle_top_y: action = Actions.DOWN.value

    return action

# TODO: set this preprocessor pipeline as default
trajectories = run_env(
    env,
    preprocessor=lambda x: extract_features(preprocess_frame(x)),
    policy=lambda x: manual_policy(x, threshold = 0.0)
)
render_video(collect_frames_fn(trajectories, "obs_hash"))

We got a reward of $-20$, meaning we scored one point. Let's bump up the threshold to 10%:

In [ ]:
trajectories = run_env(
    env,
    preprocessor=lambda x: extract_features(preprocess_frame(x)),
    policy=lambda x: manual_policy(x, threshold = 0.1)
)
frames = collect_frames_fn(trajectories, "obs_hash")
render_video(frames)

The reward increased to $-8$, so the threshold strategy work. You can try boosting it higher, but word of warning, at 25%, the episode will take forever to finish.

While this policy is good enough to make the game last for a long time, an optimal policy would finish the game as quick as possible while allowing the opponent to score no points, if possible.

A policy with 10% treshold is pretty decent for a solid start, let's use that one for training our model.

## **Train supervised policy - manual**

Moving on to model training, the first thing we need to do is to build the dataset, to do so, we want to collect as many pairs of features-actions using our policy. Training the model will make it try to fit the dataset by being able to predict the correct action based on the provided features. Let's collect the data:

In [ ]:
trajectories = run_env(
    env,
    preprocessor=lambda x:extract_features(preprocess_frame(x)),
    policy=lambda x: manual_policy(x, threshold = 0.1)
)
render_video(collect_frames_fn(trajectories, "obs_hash"))

We just need one example of which action to take given an observation, so we can discard all redundant observation-action pairs:

In [ ]:
def collect_unique_steps(trajectories):
  used = {}
  unique_steps = []
  for episode in trajectories:
    for step in episode:
      obs_processed_hash = step["obs_processed_hash"]
      if not obs_processed_hash in used: unique_steps.append(step)
      used[obs_processed_hash] = step
  return unique_steps

unique_steps = collect_unique_steps(trajectories)
print(len(unique_steps))
render_video([load_obs(step["obs_hash"]) for step in unique_steps])

We can now build a dataset out of the unique steps we collected:

In [ ]:
import torch
import random

def build_dataset(steps):
    X = []
    Y = []
    for step in steps:
        obs_processed = load_obs(step["obs_processed_hash"])
        obs_processed = torch.tensor(obs_processed).float() # TODO: use dtype
        action = step["action"]
        X.append(obs_processed)
        Y.append(action)

    # Stack the list of tensors into a 2D tensor
    X = torch.stack(X) # TODO: why stack?
    Y = torch.tensor(Y)

    return X, Y

dataset = build_dataset(unique_steps)
X, Y = dataset
X.shape, Y.shape

Sanity check for how many actions of each type are present in the dataset:

In [ ]:
actions_map = {}
for index in range(len(X)):
  features = X[index]
  action = Y[index]
  actions_map[action.item()] = actions_map.get(action.item(), 0) + 1
print(actions_map)

Define the neural network model architecture for imitation learning.

In [ ]:
import torch # TODO: remove redundant imports
import torch.nn as nn

# TODO: try MLP model
# TODO: softcode for feature size and action size
# TODO: confirm that perceptron doesnt work

import torch
import torch.nn as nn

class Model(nn.Module):
    def __init__(self, layers_list, kaiming_init=True, output_weight_scale=1.0, output_bias_scale=1.0):
        super(Model, self).__init__()

        # Create the layers dynamically based on the layers_list
        layers = []
        for i in range(len(layers_list) - 1):
            layers.append(nn.Linear(layers_list[i], layers_list[i + 1]))

        # Register the layers as a ModuleList
        self.layers = nn.ModuleList(layers)

        # Initialize weights for each layer
        if kaiming_init:
          for layer in self.layers: nn.init.kaiming_uniform_(layer.weight, nonlinearity='relu')

        # Scale down the last layer's weight
        self.layers[-1].weight.data *= output_weight_scale
        self.layers[-1].bias.data *= output_bias_scale

    def forward(self, x):
        for layer in self.layers[:-1]: x = torch.relu(layer(x))
        logits = self.layers[-1](x)
        return logits

# Example usage:
# Model with 4 inputs, two hidden layers (4, 5 neurons), and 6 outputs
model = Model([4, 4, 6])
print(model)

Visualize the results.

In [ ]:
# TODO: tune initial network to have an initial uniform bias
# TODO: tune initial learning rate; change it with training
# TODO: make sure cuda works
# TODO: make run env accept running multiple episodes and aggregating stats (so we can plot them)
# TODO: train with validation loss as well
# TODO: training vs eval mode

import torch
import torch.nn.functional as F
from torch.optim import Adam
from torch.optim.lr_scheduler import LambdaLR

SEED = 42

# Check if CUDA is available and set the device
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

def train(
    dataset,              # The dataset to train on (a tuple with a list of inputs and a list of respective outputs)
    model,                # The PyTorch model
    n_steps,              # Train for N steps
    learning_rate=0.001,  # The scale at which gradients should be applied to the parameters at each step
    batch_size=64,        # The size of the mini-batch to randomly sample in each step
    log_steps=1000,       # How frequently should training progress be logged
    seed=SEED             # The RNG seed to use when sampling mini-batches (for reproducibility)
):
    # Unpack dataset
    X, Y = dataset

    # Ensure model is on the correct device (CPU or GPU)
    model.to(DEVICE)

    # Set up the Adam optimizer
    adam_kwargs = {"lr" : learning_rate} if not callable(learning_rate) else {}
    optimizer = Adam(model.parameters(), **adam_kwargs)

    scheduler = LambdaLR(optimizer, learning_rate) if callable(learning_rate) else None

    # List to track loss
    losses = []

    # Set the random seed for reproducibility
    generator = torch.Generator().manual_seed(seed)

    for step in range(n_steps):
        # Sample a mini-batch of data
        batch_indexes = torch.randint(0, X.shape[0], (batch_size,), generator=generator)
        Xbt, Ybt = X[batch_indexes].to(DEVICE), Y[batch_indexes].to(DEVICE)

        # Forward pass now outputs probabilities (since softmax is inside forward()) # TODO: fix this comment
        logits = model(Xbt)
        loss = F.cross_entropy(logits, Ybt)

        # Zero the gradients from the previous step
        optimizer.zero_grad()

        # Perform backward pass (compute gradients)
        loss.backward()

        # Update the parameters using Adam optimizer
        optimizer.step()

        # Step the learning rate scheduler
        if scheduler: scheduler.step()

        # Track stats
        if step % log_steps == 0 or step == n_steps - 1:
            print(f"{step+1:7d}/{n_steps:7d}: {loss.item():.4f}")

        # Store the loss value
        losses.append(loss.item())

    return losses

# Create a perceptron and train it
model = Model([4, 6])
losses = train(dataset, model, 1000, learning_rate=0.01)
plt.plot(losses)

Notice the hockey stick loss? This is happening because the model's initial predictions are not uniform. The model can make quick wins by just adjusting the output layer biases. Let's make the initial distribution

In [ ]:
model = Model([4, 6])
logits = model(torch.randn((1, 4)))
logits, logits.exp(), logits.exp() / logits.exp().sum(axis=1)

Notice how the initial distribution is far from uniform. If make the logits smaller we can make it more uniform:

In [ ]:
(logits * 0.01).exp() / (logits * 0.01).exp().sum(axis=1)

Let's try again with an output layer with an initial scaled down output layer:

In [ ]:
model = Model([4, 6] , output_weight_scale=0.001, output_bias_scale=0.001)
logits = model(torch.randn((1, 4)))
logits.exp() / logits.exp().sum(axis=1)

Visualize the results.

In [ ]:
model = Model([4, 6], output_weight_scale=0.001, output_bias_scale=0.001)
losses = train(dataset, model, 1000, learning_rate=0.01)
plt.plot(losses)

Let's train it longer:

In [ ]:
model = Model([4, 6], output_weight_scale=0.001, output_bias_scale=0.001)
losses = train(dataset, model, 10_000, learning_rate=0.01)
plt.plot(losses)

Convergence has stagnated quickly, with loss jumping back and forth by the end. Let's try reducing the learning rate:

In [ ]:
losses = train(dataset, model, 10_000, learning_rate=0.001)
plt.plot(losses)

Reducing the learning rate resulted in no further improvements. We may have hit the limit of what the model can learn. Let's test it:

In [ ]:
@torch.no_grad()
def supervised_policy(model, features): # TODO: explain all this
  features = torch.tensor(features).float()
  features = features.view(1, -1)
  logits = model(features)
  action_probs = torch.softmax(logits, dim=1)
  action = torch.argmax(action_probs)
  return action

def _test_model(layers, n_training_steps=10_000):
  # TODO: add to stats if user won or not, allow aggregating average episode stats
  # TODO: make sure we test this on a different env
  # TODO: create singleton that has confi so we can configure default preprocessor
  env = make_env()
  model = Model(layers, output_weight_scale=0.001, output_bias_scale=0.001)
  losses = train(dataset, model, n_training_steps, learning_rate=0.01)
  plt.plot(losses)
  plt.show()
  trajectories = run_env(
      env,
      preprocessor=lambda x:extract_features(preprocess_frame(x)),
      policy=lambda features: supervised_policy(model, features)
  )
  frames = collect_frames_fn(trajectories, "obs_hash")
  video = render_video(frames)
  return video

_test_model([4, 6])

Notice how it did something, but quickly got stuck in a sequence of observations that it didn't know how to deal with. We need the model to learn further. Our current model is a perceptron, which means it connects the inputs directly to the outputs. All features are multiplied by a parameter, the results get added up, and that will give out the number for each action that will result in its probability. This forces the model to make all its decisions directly from the features, without any intermediate step. So it can for example say that when the linear combination of paddle and ball positions is above different tresholds, each action has different probabilities. It has no ability to first determine if the ball is above or below the paddle and then use that information to determine which action should be invoked. It has no ability abstract information. To do add that ability we need to add another layer. Let's try that out:

In [ ]:
_test_model([4, 4, 6])

Now we got exactly the same reward as our policy -8.0, the game play looks very similar as well. It's likely that the model has overfit the policy since with has the same number of hidden nuerons as input neurons, its not forced to compress information, it can route features directly to the output layer if necessary. Let's force it to reduce dimensionality:

In [ ]:
_test_model([4, 3, 6])

Interestingly with less dimensions we got a better overall episode reward. It's hard to tell if the model is overfitting or not because we don't have a validation set. So this improvement in performance may have happened due to better generalization or sheer luck. Let's lower it even more:

In [ ]:
_test_model([4, 2, 6])

Less neurons resulted in the model struggling to learn, a less total reward. We can speculate that with 4 hidden neurons, the model overfit the data. With 3 neurons it generalized a bit due to being forced to compress the features, and with 2 neurons it struggled to fit the data, but still tracks the ball. Our manual policy really only had two features, if the ball was above or below the paddle's center space. This could really be represented as just one, in the form a hiddne neuronn activating or not. So in theory it should be possible to map tha policy wih a single hidden layer.

In [ ]:
_test_model([4, 1, 6])

Run the experiment with different configurations.

In [ ]:
# TODO: create experiment tracker that trains N models and plots all their loss curves and best losses
# TODO: make sure we use generators for everything, including model init
env = make_env()
model = Model([4, 1, 6], output_weight_scale=0.001, output_bias_scale=0.001)
losses = train(dataset, model, 10_000, learning_rate=0.01)

Run the following analysis.

In [ ]:
losses = train(dataset, model, 10_000, learning_rate=0.001, batch_size=128)

Visualize the results.

In [ ]:
plt.plot(losses)
plt.show()
trajectories = run_env(
    env,
    preprocessor=lambda x:extract_features(preprocess_frame(x)),
    policy=lambda features: supervised_policy(model, features)
)
frames = collect_frames_fn(trajectories, "obs_hash")
render_video(frames)

Compare the different approaches.

In [ ]:
# TODO: log experiment id
# TODO: comment code
# TODO: log env details
# TODO: speculate about outcomes
# TODO: specify that non deterministic envs will have different outcomes each run
# TODO: make run env stats also return action stats

env_ids = [
    "PongNoFrameskip-v4",    # frameskip=1, repeat_action_probability=0.0
    "PongDeterministic-v4",  # frameskip=4, repeat_action_probability=0.0
    "PongNoFrameskip-v0",    # frameskip=1, repeat_action_probability=0.25
    "Pong-v4",               # frameskip=(2,4), repeat_action_probability=0.0
    "ALE/Pong-v5"            # frameskip=4, repeat_action_probability=0.25
]

videos = []
for env_id in env_ids:
  env = make_env(env_id)
  trajectories = run_env(
      env,
      preprocessor=lambda x:extract_features(preprocess_frame(x)),
      policy=lambda features: supervised_policy(model, features)
  )
  frames = collect_frames_fn(trajectories, "obs_hash")
  videos.append((frames, env_id))
render_videos(videos)

Run the following analysis.

In [ ]:
# TODO: add noop max?
# TODO: implement this ourselves?
# TODO: fix generalization using validation set and dropout?
# TODO: save resulting model for next lesson

## THE END

Check my repo for more **AI/ML** notebooks: https://github.com/tsilva/aiml-notebooks